# 微專題 03：用 Colab 訓練剪刀石頭布模型（選用）

自己拍的照片不多，一般筆電用 CPU 就訓練得完。想用比較大的 `imgsz=640` 練 50 輪、電腦又沒有顯示卡的話，可以借 Google 的 GPU。

**開始前**

1. 在自己的電腦上執行完 `uv run rps_data.py`，課程資料夾裡會有 `datasets/rps` 資料夾。
2. 把 `datasets/rps` 這個資料夾壓縮成 zip：Mac 在 Finder 對它按右鍵 →「壓縮」；Windows 按右鍵 →「壓縮至」→「ZIP 檔案」。
3. 這本筆記本的選單「執行階段 → 變更執行階段類型」，硬體加速器選 **T4 GPU**，按「儲存」。

然後從上到下，一格一格按 ▶ 執行，等上一格跑完再按下一格。Colab 斷線或關掉以後，裡面的檔案會全部清掉，訓練完記得馬上下載 `best.pt`。

## 1. 安裝 ultralytics

Colab 已經裝好 PyTorch 和 OpenCV，只要再裝 ultralytics，大約 1 分鐘。

In [ ]:
%pip install -q ultralytics

## 2. 上傳資料集，寫出 rps.yaml

按下執行後，下面會出現「選擇檔案」按鈕，選剛才壓縮的 zip。程式會解壓縮、找出 train 和 valid 資料夾，寫出 `rps.yaml`，再印出每一組有幾張照片、每一類有幾個框。

In [ ]:
import shutil
import zipfile
from pathlib import Path

from google.colab import files

NAMES = ["rock", "paper", "scissors"]  # 類別編號 0、1、2，和 rps_data.py、rps_game.py 一樣

uploaded = files.upload()  # 跳出「選擇檔案」
zip_name = next(iter(uploaded))
shutil.rmtree("upload", ignore_errors=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall("upload")

# zip 裡可能多包一層資料夾，找到有 train/images 的那一層；Mac 壓縮時多出來的 __MACOSX 不算
train_dir = next(p for p in Path("upload").rglob("train/images") if "__MACOSX" not in p.parts).parent.parent
data_dir = Path("datasets/rps")
shutil.rmtree(data_dir, ignore_errors=True)
data_dir.parent.mkdir(exist_ok=True)
shutil.move(str(train_dir), data_dir)

lines = [f'path: "{data_dir.resolve().as_posix()}"', "train: train/images", "val: valid/images", "names:"]
lines += [f"  {i}: {name}" for i, name in enumerate(NAMES)]
Path("rps.yaml").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("已寫出 rps.yaml")

for split in ["train", "valid"]:
    images = list((data_dir / split / "images").glob("*.jpg"))
    boxes = [0] * len(NAMES)
    for txt in (data_dir / split / "labels").glob("*.txt"):
        for line in txt.read_text().splitlines():
            if line.strip():
                boxes[int(line.split()[0])] += 1  # 每行第一個數字是類別編號
    detail = "  ".join(f"{name} {n}" for name, n in zip(NAMES, boxes))
    print(f"{split:<6}{len(images):>5} 張  框 {sum(boxes)} 個：{detail}")

## 3. 確認有 GPU

印出 GPU 的名字（例如 Tesla T4）才往下做。印出「沒有 GPU」的話，照最上面的說明換成 T4 GPU；換了以後檔案會清掉，要從第 1 格重新執行。

In [ ]:
import torch

if torch.cuda.is_available():
    print("有 GPU：", torch.cuda.get_device_name(0))
else:
    print("沒有 GPU：到「執行階段 → 變更執行階段類型」選 T4 GPU，再從第 1 格重新執行。")
    print("用 CPU 也能訓練，但會慢很多。")

## 4. 訓練

和在自己電腦上打的指令一樣：

- `model=yolo26n.pt`：從官方訓練好的 yolo26n 接著訓練，第一次會自動下載。
- `data=rps.yaml`：上一格寫出的資料集設定。
- `epochs=50`：訓練照片全部看 50 輪。
- `imgsz=640`：照片縮成 640×640 再訓練。

每一輪印一行進度，接著用驗證集打分數（`mAP50`、`mAP50-95`，越接近 1 越好）。全部跑完，最後幾行會寫結果存在哪個資料夾，通常是 `runs/detect/train`。

In [ ]:
!yolo train model=yolo26n.pt data=rps.yaml epochs=50 imgsz=640

## 5. 看訓練結果

顯示最新一次訓練的兩張圖：

- `results.png`：損失（loss）和分數隨著輪數的變化。損失往下、分數往上，最後變平，就是學得差不多了。
- `val_batch0_pred.jpg`：驗證照片上的預測結果。

In [ ]:
from pathlib import Path

from IPython.display import Image, display

run = max(Path("runs/detect").glob("train*"), key=lambda p: p.stat().st_mtime)  # 最新的一次訓練
print("訓練結果在", run)
display(Image(filename=str(run / "results.png"), width=900))
display(Image(filename=str(run / "val_batch0_pred.jpg"), width=600))

## 6. 把 best.pt 下載回電腦

`best.pt` 是驗證分數最好那一輪的權重。瀏覽器會把它存到「下載」資料夾，把它移到課程資料夾，再執行：

```sh
uv run rps_game.py
```

瀏覽器沒有跳出下載的話：點左邊的資料夾圖示，打開 `runs/detect/train/weights`（第二次訓練是 `train-2`），在 `best.pt` 上按右鍵 →「下載」。

In [ ]:
from pathlib import Path

from google.colab import files

run = max(Path("runs/detect").glob("train*"), key=lambda p: p.stat().st_mtime)  # 最新的一次訓練
best = run / "weights" / "best.pt"
print("下載", best)
files.download(str(best))